# Generation of ketamine data pair

In [16]:
%load_ext autoreload
%autoreload 2
%matplotlib tk
import numpy as np
import matplotlib.pyplot as plt
from Functions.generate_OU import get_mixed_OU_signals
from Functions.time_frequency import spectrogram
from scipy.linalg import solve

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
def whittaker_als_baseline(y, lam=1e4, p=0.01, max_iter=15):
    L = len(y)
    D = np.zeros((L-2, L))
    for i in range(L-2):
        D[i, i], D[i, i+1], D[i, i+2] = 1, -2, 1
    penalty = lam * np.dot(D.T, D)
    w = np.ones(L)
    for _ in range(max_iter):
        W = np.diag(w)
        z = solve(W + penalty, w * y)
        w = np.where(y > z, p, 1 - p)
    return z

## Inspect how to generate synthetic data pairs

In [10]:
# --- Parameters
T = 20 # desired signal duration (s)
dt = 0.001
fs = 1 / dt

lbda_list = [1, 2]
omega_list = [2*np.pi*1, 2*np.pi*10]
sigma_list = [3, 2]
factor_list = [1, 1]

# --- Generate EEG data
t, y = get_mixed_OU_signals(T, dt, lbda_list, omega_list, sigma_list, factor_list)

# --- compute spectrogram
f_spectro, t_spectro, spectro = spectrogram(y, fs, nfft_factor=2)

# --- Display
fig, axes = plt.subplots(3,  constrained_layout = True)
axes[0].plot(t, y)
axes[0].set_title('Simulated EEG signal')
axes[1].pcolormesh(t_spectro, f_spectro, np.log2(spectro + 1e-11), shading = 'nearest', cmap = 'jet')
axes[1].set_title('Spectrogram')
axes[2].plot(f_spectro, np.log2(np.median(spectro, axis = 1)))
axes[2].set_title('PSD')
axes[1].sharex(axes[0])

plt.show()

__Take part of signal to enhance__

In [20]:
mask_f = f_spectro >= 20
f_M = f_spectro[mask_f]
M = spectro[mask_f, :].copy()

psd = np.mean(M, axis = 1)
psd_baseline = whittaker_als_baseline(psd, lam=1e1, p=0.01)

# --- Display
fig, axes = plt.subplots(2,  constrained_layout = True)
axes[0].pcolormesh(t_spectro, f_M, np.log2(M + 1e-11), shading = 'nearest', cmap = 'jet')
axes[0].set_title('Spectrogram')
axes[1].plot(f_M, psd)
axes[1].plot(f_M, psd_baseline)
axes[1].set_title('PSD')

plt.show()

## 1. select bins to enhance

In [ ]:
# --- Segmentation using PSD baseline
factor_high = 3
T_high = factor_high * psd_baseline  # Using your defined factor
mask = M > (T_high[:, np.newaxis]) 


# --- 1. Your Existing Mask Setup ---
f_int = [22, 38]

freq_mask = (f_M >= f_int[0]) & (f_M <= f_int[-1])
mask_f_int = mask * freq_mask[:, None]

fig, axes = plt.subplots(3, sharex = True, constrained_layout = True)
axes[0].pcolormesh(t_spectro, f_M, np.log2(M + 1e-11), shading = 'nearest', cmap = 'jet')
axes[1].pcolormesh(t_spectro, f_M, mask, shading = 'nearest', cmap = 'jet')
axes[2].pcolormesh(t_spectro, f_M, np.log2(M + 1e-11), shading = 'nearest', cmap = 'jet')
axes[2].contour(t_spectro, f_M, mask, colors = 'white')

plt.show()

In [55]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.segmentation import watershed

# --- 1. Define Target Frequency Band (100% Selected as Seed) ---
f_int = [25, 35]
freq_mask = (f_M >= f_int[0]) & (f_M <= f_int[-1])

# Create seed mask: Entire band [22, 38] Hz is selected
seed_mask = np.tile(freq_mask[:, None], (1, M.shape[1]))

# --- 2. Define High-Intensity Threshold for Expansion ---
psd = np.mean(M, axis=1)
psd_baseline = whittaker_als_baseline(psd, lam=1e4, p=0.01)

# Pixels outside the band must pass this threshold to be flooded into
expansion_threshold = psd_baseline[:, np.newaxis] * 1.5  # Adjust multiplier (1.8 - 3.0)
high_intensity_mask = M > expansion_threshold

# --- 3. Prepare Watershed Markers & Allowable Region ---
# Allow flooding ANYWHERE inside the band OR inside high-intensity regions outside
allowable_region = seed_mask | high_intensity_mask

markers = np.zeros(M.shape, dtype=int)

# Seed 1: The target band (where flooding starts)
markers[seed_mask] = 1

# Background (-1): Areas below the high-intensity threshold outside the band (stopping boundary)
markers[~allowable_region] = -1

# --- 4. Run Watershed Expansion ---
# Floods outward from seed_mask across the inverted intensity surface (-M)
inverted_M = -M
labels = watershed(inverted_M, markers, mask=allowable_region)

# Final expanded mask (contains the band + watershed-flooded connected regions)
expanded_mask = labels > 0

# --- 5. Visualization ---
fig, axes = plt.subplots(3, 1, sharex=True, constrained_layout=True)

# Spectrogram + Base Band Highlight
axes[0].pcolormesh(t_spectro, f_M, np.log2(M + 1e-11), shading='nearest', cmap='jet')
axes[0].axhline(f_int[0], color='white', linestyle='--', alpha=0.7)
axes[0].axhline(f_int[1], color='white', linestyle='--', alpha=0.7)
axes[0].set_ylim(f_M.min(), f_M.max())
axes[0].set_xlim(t_spectro.min(), t_spectro.max())
axes[0].set_title(f"Base Seed Band [{f_int[0]}, {f_int[1]}] Hz")

# Watershed Expanded Mask
axes[1].pcolormesh(t_spectro, f_M, expanded_mask, shading='nearest', cmap='jet')
axes[1].set_title("Watershed Expanded Mask")

# Overlay Contour
axes[2].pcolormesh(t_spectro, f_M, np.log2(M + 1e-11), shading='nearest', cmap='jet')
axes[2].contour(t_spectro, f_M, expanded_mask, colors='white', linewidths=0.75)
axes[2].set_title("Watershed Overlay on Spectrogram")

plt.show()

In [60]:
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Extract Selected Pixels & Compute Empirical Quantile Ranks ---
selected_data = M[expanded_mask]

# Compute quantile rank (0 to 1) for every pixel in the selected mask
# Using searchsorted on sorted data gives precise empirical CDF ranks
sorted_vals = np.sort(selected_data)
ranks = np.searchsorted(sorted_vals, selected_data, side='right') / len(selected_data)

# --- 2. Set Target Parameters ---
old_min = selected_data.min()
old_max = selected_data.max()

# Define your desired NEW maximum (e.g., 3x or 5x the old maximum)
target_max = old_max * 3.0  # Adjust scaling factor as needed!

# Convex exponent: 
# alpha = 1.0 -> Linear stretch across all values
# alpha > 1.0 (e.g. 2.0 to 4.0) -> Keeps low values near old_min, heavily inflates top quantiles
alpha = 2.5 

# --- 3. Compute Quantile-Adapted Values ---
transformed_selected = old_min + (target_max - old_min) * (ranks ** alpha)

# Apply back to the matrix
M_transformed = M.copy()
M_transformed[expanded_mask] = transformed_selected

# --- 4. Visualization & Verification ---
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)

# A. Original Spectrogram
axes[0, 0].pcolormesh(t_spectro, f_M, np.log2(M + 1e-11), shading='nearest', cmap='jet')
axes[0, 0].set_title(f"Original Spectrogram (Max = {old_max:.2e})")

# B. Transformed Spectrogram
axes[0, 1].pcolormesh(t_spectro, f_M, np.log2(M_transformed + 1e-11), shading='nearest', cmap='jet')
axes[0, 1].set_title(f"Quantile-Adapted Spectrogram (New Max = {target_max:.2e})")

# C. Empirical Quantile vs New Values
sorted_transformed = np.sort(transformed_selected)
cdf_q = np.linspace(0, 1, len(sorted_vals))

axes[1, 0].plot(cdf_q, sorted_vals, label="Original Quantiles", color="blue")
axes[1, 0].plot(cdf_q, sorted_transformed, label="Target Quantiles", color="red", linestyle="--")
axes[1, 0].set_xlabel("Quantile Rank (q)")
axes[1, 0].set_ylabel("Pixel Intensity")
axes[1, 0].set_title("Quantile Function Mapping")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# D. Old Intensity vs Transformed Intensity
axes[1, 1].scatter(selected_data, transformed_selected, s=1, color="purple", alpha=0.3)
axes[1, 1].plot([old_min, old_max], [old_min, old_max], 'k--', alpha=0.5, label="1:1 Identity Line")
axes[1, 1].set_xlabel("Old Intensity")
axes[1, 1].set_ylabel("New Transformed Intensity")
axes[1, 1].set_title("Transfer Curve (Old vs New)")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.show()

In [61]:
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Define Target Frequency Bump PSD ---
psd_orig = np.mean(M, axis=1)

# Create mid-frequency bump centered at f_center
f_int = [22, 38]
f_center = np.mean(f_int)
f_sigma = (f_int[1] - f_int[0]) / 4.0

# Gaussian profile to shape target PSD
gaussian_bump = np.exp(-0.5 * ((f_M - f_center) / f_sigma) ** 2)

# Target PSD: Original decay baseline + targeted bump boost (e.g., 3x boost)
target_psd = psd_orig * (1.0 + 3.0 * gaussian_bump)

# --- 2. Construct Target Intensity Matrix Distribution ---
# Generate a synthetic target matrix with the new frequency shape
# We scale time slices to mimic target PSD while maintaining variance
M_target_distribution = M * (target_psd[:, np.newaxis] / (psd_orig[:, np.newaxis] + 1e-12))

# --- 3. Exact Rank/Histogram Matching ---
# Flatten matrices to operate on ranks
M_flat = M.flatten()
target_flat = M_target_distribution.flatten()

# Get sorting order of original matrix M
sort_order = np.argsort(M_flat)

# Sort target values in ascending order
sorted_target_vals = np.sort(target_flat)

# Map target values to M based on original ranks
M_matched_flat = np.empty_like(M_flat)
M_matched_flat[sort_order] = sorted_target_vals

# Reshape back to original Spectrogram dimensions
M_matched = M_matched_flat.reshape(M.shape)

# --- 4. Verification & Visualization ---
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)

# A. Original Spectrogram
axes[0, 0].pcolormesh(t_spectro, f_M, np.log2(M + 1e-11), shading='nearest', cmap='jet')
axes[0, 0].set_title("Original Spectrogram M")

# B. Rank-Matched Spectrogram
axes[0, 1].pcolormesh(t_spectro, f_M, np.log2(M_matched + 1e-11), shading='nearest', cmap='jet')
axes[0, 1].set_title("Rank-Matched Synthetic Spectrogram")

# C. PSD Comparison
axes[1, 0].plot(f_M, psd_orig, label="Original PSD", color="blue")
axes[1, 0].plot(f_M, np.mean(M_matched, axis=1), label="Matched PSD (With Bump)", color="red", linestyle="--")
axes[1, 0].axvspan(f_int[0], f_int[1], color='cyan', alpha=0.15, label='Target Band [22, 38] Hz')
axes[1, 0].set_xlabel("Frequency (Hz)")
axes[1, 0].set_ylabel("Mean Power")
axes[1, 0].set_title("PSD Profile Shaping")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# D. Rank Correlation Check
axes[1, 1].scatter(M_flat[::50], M_matched_flat[::50], s=2, color="purple", alpha=0.5)
axes[1, 1].set_xlabel("Original Intensity Rank")
axes[1, 1].set_ylabel("Matched Intensity Value")
axes[1, 1].set_title("Monotonic Transfer Curve (Strict Order Preserved)")
axes[1, 1].grid(True, alpha=0.3)

plt.show()